In [2]:
import pandas as pd
import numpy as np

# 이전 단계에서 생성한 데이터 로드
try:
    df_rfm = pd.read_csv('data/customer_rfm.csv')
    df_taste = pd.read_csv('data/customer_taste_clusters.csv')
    df_orders = pd.read_csv('data/customer_orders.csv') # 유사 제품 구매 이력 분석을 위해 원본 데이터 로드
    
    # CustomerID를 기준으로 RFM 세그먼트와 취향 클러스터를 병합하여 최종 세그먼트 데이터 생성
    df_segments = pd.merge(df_rfm, df_taste, on='CustomerID')
    
    print("✅ 최종 세그먼트 데이터 준비 완료.")
    print(f"총 고객 수: {len(df_segments)}명")
    display(df_segments.head())

except FileNotFoundError as e:
    print(f"🚨 오류: 필수 데이터 파일({e.filename})이 없습니다. 이전 단계를 먼저 실행해주세요.")

✅ 최종 세그먼트 데이터 준비 완료.
총 고객 수: 996명


,CustomerID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,Segment,Taste_Cluster
0,C0001,86,4,180000,3,2,3,323,잠재 고객 (Potential),2
1,C0002,99,5,75000,3,3,1,331,충성 고객 (Loyal),0
2,C0003,199,4,121000,2,2,2,222,이탈 위험 고객 (At Risk),0
3,C0004,170,2,71000,2,1,1,211,이탈 위험 고객 (At Risk),3
4,C0005,116,4,155000,3,2,3,323,잠재 고객 (Potential),2


In [9]:
# --- 1. 신제품 정보 정의 ---
# 컨조인트 시뮬레이션에서 페르소나가 선택한 옵션을 기반으로 설정
NEW_PRODUCT_PRICE = 20000 # 예상 구매액 (가격)
NEW_PRODUCT_NAME = "데일리 에너지업 알약 비타민"

# --- 2. 유사 제품 카테고리 정의 ---
SIMILAR_PRODUCT_CATEGORY = '건강기능식품'

# --- 3. 유사 제품을 구매한 이력이 있는 고객 ID 목록 생성 ---
# df_orders에서 '건강기능식품' 카테고리를 구매한 모든 고객의 ID를 중복 없이 추출
similar_product_buyers = set(df_orders[df_orders['ProductCategory'] == SIMILAR_PRODUCT_CATEGORY]['CustomerID'])

print(f"신제품 '{NEW_PRODUCT_NAME}' (가격: {NEW_PRODUCT_PRICE:,}원)에 대한 매출 예측을 시작합니다.")
print(f"유사 제품군 '{SIMILAR_PRODUCT_CATEGORY}'을 구매한 이력이 있는 고객은 총 {len(similar_product_buyers)}명입니다.")

신제품 '데일리 에너지업 알약 비타민' (가격: 20,000원)에 대한 매출 예측을 시작합니다.
유사 제품군 '건강기능식품'을 구매한 이력이 있는 고객은 총 861명입니다.


In [10]:
# 각 고객이 유사 제품을 구매했는지 여부를 True/False로 표시하는 컬럼 추가
df_segments['bought_similar'] = df_segments['CustomerID'].isin(similar_product_buyers)

# 세그먼트별로 그룹화하여 '고객 수'와 '구매 전환율'을 한번에 계산
# 'bought_similar' 컬럼(True=1, False=0)의 평균을 구하면 그것이 바로 구매율이 됨
prediction_df = df_segments.groupby(['Segment', 'Taste_Cluster']).agg(
    Segment_Size=('CustomerID', 'count'),
    Adoption_Rate=('bought_similar', 'mean')
).reset_index()

print("✅ 세그먼트별 예상 초기 구매 전환율 계산 완료:")
display(prediction_df)

✅ 세그먼트별 예상 초기 구매 전환율 계산 완료:


,Segment,Taste_Cluster,Segment_Size,Adoption_Rate
0,VIP,0,48,0.958333
1,VIP,1,69,1.000000
2,VIP,2,29,0.965517
3,VIP,3,52,0.980769
4,이탈 위험 고객 (At Risk),0,66,0.621212
5,이탈 위험 고객 (At Risk),1,67,1.000000
6,이탈 위험 고객 (At Risk),2,55,0.636364
7,이탈 위험 고객 (At Risk),3,64,0.578125
8,잠재 고객 (Potential),0,84,0.833333
9,잠재 고객 (Potential),1,99,1.000000


In [11]:
# --- 1. 예상 구매액(가격) 컬럼 추가 ---
prediction_df['Expected_Price'] = NEW_PRODUCT_PRICE

# --- 2. 세그먼트별 예측 매출 계산 ---
prediction_df['Segment_Forecast'] = prediction_df['Segment_Size'] * prediction_df['Adoption_Rate'] * prediction_df['Expected_Price']

# --- 3. 최종 결과 정렬 및 출력 ---
# 예상 매출액이 높은 순으로 정렬하여 어떤 세그먼트가 핵심 타겟인지 확인
prediction_df_sorted = prediction_df.sort_values('Segment_Forecast', ascending=False)

print(f"✅ 신제품 '{NEW_PRODUCT_NAME}'의 세그먼트별 예측 매출:")
# 보기 좋게 포맷팅
prediction_df_sorted['Segment_Forecast'] = prediction_df_sorted['Segment_Forecast'].apply(lambda x: f"{int(x):,}원")
display(prediction_df_sorted[['Segment', 'Taste_Cluster', 'Segment_Size', 'Adoption_Rate', 'Segment_Forecast']])

# --- 4. 총 예측 매출 계산 ---
total_predicted_revenue = prediction_df['Segment_Forecast'].sum()

print("\n" + "="*60)
print("🎉 최종 예측 결과 🎉")
print(f"신제품 '{NEW_PRODUCT_NAME}' 출시 초기, 예상되는 총 매출액은...")
print(f"약 {int(total_predicted_revenue):,} 원 입니다.")
print("="*60)

✅ 신제품 '데일리 에너지업 알약 비타민'의 세그먼트별 예측 매출:


,Segment,Taste_Cluster,Segment_Size,Adoption_Rate,Segment_Forecast
9,잠재 고객 (Potential),1,99,1.000000,"1,980,000원"
8,잠재 고객 (Potential),0,84,0.833333,"1,400,000원"
1,VIP,1,69,1.000000,"1,380,000원"
5,이탈 위험 고객 (At Risk),1,67,1.000000,"1,340,000원"
11,잠재 고객 (Potential),3,77,0.818182,"1,260,000원"
13,충성 고객 (Loyal),1,63,1.000000,"1,260,000원"
10,잠재 고객 (Potential),2,80,0.762500,"1,220,000원"
3,VIP,3,52,0.980769,"1,020,000원"
0,VIP,0,48,0.958333,"920,000원"
14,충성 고객 (Loyal),2,47,0.978723,"920,000원"



🎉 최종 예측 결과 🎉
신제품 '데일리 에너지업 알약 비타민' 출시 초기, 예상되는 총 매출액은...
약 17,220,000 원 입니다.
